# Seydoux et al. (2022) NPQ model for diatoms

**Authors:** Claire Seydoux, Mattia Storti, Vasco Giovagnetti, Anna Barbara Matuszyńska, Erika Guglielmino, Xue Zhao, Cécile Giustini, Yufang Pan, Lander Blommaert, Jhoanell Angulo, Alexander V. Ruban, Hanhua Hu, Benjamin Bailleul, Florence Courtois, Guillaume Allorent, and Giovanni Finazzi

**Title:** *Impaired photoprotection in Phaeodactylum tricornutum KEA3 mutants reveals the proton regulatory circuit of diatoms light acclimation*

**Journal:** *New Phytologist* (2022)  
**DOI:** 10.1111/nph.18003  
**Organism:** *Phaeodactylum tricornutum*

## Paper summary
The model contains five dynamic variables: reduced plastoquinone, lumenal
protons, active ATP synthase, ATP, and diadinoxanthin. NPQ is calculated from
PSII fluorescence and the diatoxanthin-dependent quencher. 
The model investigates how KEA3-dependent partitioning of the proton-motive force controls lumen acidification, the diadinoxanthin–diatoxanthin cycle, and non-photochemical quenching in diatoms.

This notebook recreates only the modelling panels:

- **Figure 3b:** NPQ kinetics during illumination at 200 µmol photons m⁻² s⁻¹ and subsequent dark recovery.
- **Figure 3c:** NPQ as a function of actinic-light intensity.

The equations and parameter values are translated from the authors' `model.py`, while the alternating light and saturating-pulse sequence follows `functions.py` and `run_pam.py`.

## 1. Imports

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Final

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mxlpy import Derived, Model, Simulator
from mxlpy.surrogates import qss

## 2. Genotype conditions and experiment constants

The original implementation changes the fraction of the proton-motive force assigned to the KEA3-dependent component:

- KEA3 mutant: `10.0`
- WT: `2.0`
- KEA3 overexpressor/complemented line: `1.8`

In [ ]:
@dataclass(frozen=True)
class Genotype:
    label: str
    k_kea_fraction: float
    color: str

    @property
    def pmf_ratio(self) -> float:
        return self.k_kea_fraction / (1.0 - self.k_kea_fraction)


GENOTYPES: Final[dict[str, Genotype]] = {
    "Mutant 1": Genotype("KEA3 mutant", 0.865, "limegreen"),
    "WT": Genotype("WT", 0.700, "red"),
    "Overexpressor 1": Genotype("KEA3 overexpressor", 0.650, "blue"),
}

SATURATING_PULSE: Final[float] = 5000.0
ACTINIC_LIGHT: Final[float] = 200.0
RECOVERY_LIGHT: Final[float] = 0.0

## 3. Model equations

The dynamic variables are reduced plastoquinone, lumenal protons, ATP-synthase activity, ATP, and diadinoxanthin. The following helper functions define thermodynamic relationships, PSII states, fluorescence, and reaction rates.

In [ ]:
# ------------------------------------------------------------------
# Model equations
# ------------------------------------------------------------------
def _ph(protons: float) -> float:
    return -np.log10(protons * 3.2e-5)


def _ph_inv(ph: float) -> float:
    return 3.125e4 * 10.0 ** (-ph)


def _moiety(value: float, total: float) -> float:
    return total - value


def _scaled_protons(protons: float, k_kea_fraction: float) -> float:
    return k_kea_fraction * protons


def _quencher(dd: float, x_total: float, lhcx_total: float) -> float:
    dt = x_total - dd
    return lhcx_total * dt


def _keq_qa_pq(
    faraday: float,
    e0_qa: float,
    e0_pq: float,
    ph_stroma: float,
    gas_constant: float,
    temperature: float,
) -> float:
    rt = gas_constant * temperature
    dg1 = -faraday * e0_qa
    dg2 = -2.0 * faraday * e0_pq + 2.0 * ph_stroma * np.log(10.0) * rt
    dg0 = -2.0 * dg1 + dg2
    return np.exp(-dg0 / rt)


def _keq_cytb6f(
    h_beta: float,
    faraday: float,
    e0_pq: float,
    gas_constant: float,
    temperature: float,
    e0_pc: float,
    ph_stroma: float,
) -> float:
    rt = gas_constant * temperature
    ph_lumen = _ph(h_beta)
    dg1 = -2.0 * faraday * e0_pq + 2.0 * rt * np.log(10.0) * ph_lumen
    dg2 = -faraday * e0_pc
    dg3 = rt * np.log(10.0) * (ph_stroma - ph_lumen)
    dg = -dg1 + 2.0 * dg2 + 2.0 * dg3
    return np.exp(-dg / rt)


def _keq_atp_synthase(
    protons: float,
    dg0_atp: float,
    ph_stroma: float,
    gas_constant: float,
    temperature: float,
    phosphate: float,
) -> float:
    rt = gas_constant * temperature
    dg = dg0_atp - np.log(10.0) * (ph_stroma - _ph(protons)) * (14.0 / 3.0) * rt
    return phosphate * np.exp(-dg / rt)


def _psii_states(
    pq_red: float,
    quencher: float,
    ppfd: float,
    pq_total: float,
    k_pq_red: float,
    keq_qa_pq: float,
    k_h: float,
    k_f: float,
    k_p: float,
    psii_total: float,
) -> tuple[float, float, float, float]:
    pq_ox = pq_total - pq_red
    b0 = ppfd + k_pq_red * pq_red / keq_qa_pq
    b1 = k_h * quencher + k_f
    b2 = k_h * quencher + k_f + k_p

    matrix = np.array(
        [
            [-b0, b1, k_pq_red * pq_ox, 0.0],
            [ppfd, -b2, 0.0, 0.0],
            [0.0, 0.0, ppfd, -b1],
            [1.0, 1.0, 1.0, 1.0],
        ],
        dtype=float,
    )
    rhs = np.array([0.0, 0.0, 0.0, psii_total], dtype=float)
    states = np.linalg.solve(matrix, rhs)
    return tuple(float(value) for value in states)


def _fluorescence(
    quencher: float,
    b0: float,
    b2: float,
    k_h: float,
    k_f: float,
    k_p: float,
) -> float:
    return k_f / (k_h * quencher + k_f + k_p) * b0 + k_f / (k_h * quencher + k_f) * b2


def _v_psii(b1: float, k_p: float) -> float:
    return 0.5 * k_p * b1


def _v_pq_oxidation(
    pq_red: float,
    ppfd: float,
    k_cytb6f: float,
    pq_total: float,
    keq_cytb6f: float,
) -> float:
    k_light = k_cytb6f * ppfd
    forward = k_light * keq_cytb6f / (keq_cytb6f + 1.0)
    reverse = k_light / (keq_cytb6f + 1.0)
    return forward * pq_red - reverse * (pq_total - pq_red)


def _v_atpase_activity(
    activity: float,
    ppfd: float,
    k_activation: float,
    k_deactivation: float,
) -> float:
    if ppfd > 0.0:
        return k_activation * (1.0 - activity)
    return -k_deactivation * activity


def _v_atp_synthase(
    atp: float,
    activity: float,
    k_atp_synthase: float,
    keq_atp_synthase: float,
    adenylate_total: float,
) -> float:
    return activity * k_atp_synthase * (adenylate_total - atp - atp / keq_atp_synthase)


def _v_atp_consumption(atp: float, k_atp_consumption: float) -> float:
    return k_atp_consumption * atp


def _v_proton_leak(protons: float, k_leak: float, h_stroma: float) -> float:
    return k_leak * (protons - h_stroma)


def _deepoxidase_activity(
    h_beta: float,
    hill_coefficient: float,
    ph_half_saturation: float,
    k_deepoxidation: float,
) -> float:
    h_half = _ph_inv(ph_half_saturation)
    numerator = h_beta**hill_coefficient
    return k_deepoxidation * numerator / (numerator + h_half**hill_coefficient)


def _v_dd_deepoxidation(dd: float, k_deepoxidase: float) -> float:
    return k_deepoxidase * dd


def _v_dt_epoxidation(
    dt: float,
    ppfd: float,
    k_epoxidation: float,
    k_etc: float,
) -> float:
    low_light_factor = k_etc if ppfd <= 50.0 else 1.0
    return low_light_factor * k_epoxidation * dt


def _two_over(value: float) -> float:
    return 2.0 / value


def _four_over(value: float) -> float:
    return 4.0 / value


def _negative_hpr_over(hpr: float, buffer_capacity: float) -> float:
    return -hpr / buffer_capacity


def _negative_one_over(value: float) -> float:
    return -1.0 / value

## 4. Build the MxlPy model

In [ ]:
def get_seydoux2022_npq(genotype: str = "WT") -> Model:
    r"""Return the Seydoux et al. diatom NPQ model in MxlPy format."""
    if genotype not in GENOTYPES:
        valid = ", ".join(GENOTYPES)
        raise ValueError(f"Unknown genotype {genotype!r}. Choose one of: {valid}.")

    m = Model()

    m.add_variables(
        {
            "pq_red": 5.0,
            "protons": 5.56305280e-4,
            "atpase_activity": 0.5,
            "atp": 25.0,
            "dd": 0.9,
        }
    )
    m.add_parameters(
        {
            "PSII_total": 2.5,
            "PQ_total": 20.0,
            "AP_total": 50.0,
            "LHCX_total": 0.5,
            "X_total": 1.0,
            "O2_ex": 8.0,
            "Pi": 0.01,
            "k_KEA_fraction": GENOTYPES[genotype].k_kea_fraction,
            "k_Cytb6f": 0.104,
            "k_ActATPase": 0.01,
            "k_DeactATPase": 0.002,
            "k_ATPsynthase": 20.0,
            "k_ATPconsumption": 10.0,
            "k_PQred": 250.0,
            "k_H": 5.0e9,
            "k_F": 6.25e8,
            "k_P": 1.2e9,
            "pH_stroma": 7.8,
            "k_leak": 1000.0,
            "b_H": 100.0,
            "HPR": 14.0 / 3.0,
            "k_deepoxidation": 0.025,
            "k_epoxidation": 0.0096,
            "pH_deepoxidase_half": 6.3,
            "n_HX": 5.0,
            "k_ETC": 2.0,
            "F": 96.485,
            "R": 8.3e-3,
            "T": 298.0,
            "E0_QA": -0.140,
            "E0_PQ": 0.354,
            "E0_PC": 0.350,
            "DG0_ATP": 30.6,
            "PPFD": 0.0,
        }
    )

    m.add_derived("H_stroma", fn=_ph_inv, args=["pH_stroma"])
    m.add_derived("H_beta", fn=_scaled_protons, args=["protons", "k_KEA_fraction"])
    m.add_derived("pH_lumen", fn=_ph, args=["H_beta"])
    m.add_derived("pq_ox", fn=_moiety, args=["pq_red", "PQ_total"])
    m.add_derived("adp", fn=_moiety, args=["atp", "AP_total"])
    m.add_derived("dt", fn=_moiety, args=["dd", "X_total"])
    m.add_derived("Q", fn=_quencher, args=["dd", "X_total", "LHCX_total"])
    m.add_derived(
        "K_QAPQ",
        fn=_keq_qa_pq,
        args=["F", "E0_QA", "E0_PQ", "pH_stroma", "R", "T"],
    )
    m.add_derived(
        "K_cytb6f",
        fn=_keq_cytb6f,
        args=["H_beta", "F", "E0_PQ", "R", "T", "E0_PC", "pH_stroma"],
    )
    m.add_derived(
        "K_ATPsynthase",
        fn=_keq_atp_synthase,
        args=["protons", "DG0_ATP", "pH_stroma", "R", "T", "Pi"],
    )
    m.add_derived(
        "k_deepoxidase",
        fn=_deepoxidase_activity,
        args=["H_beta", "n_HX", "pH_deepoxidase_half", "k_deepoxidation"],
    )

    m.add_surrogate(
        "PSII_states",
        qss.Surrogate(
            model=_psii_states,
            args=[
                "pq_red",
                "Q",
                "PPFD",
                "PQ_total",
                "k_PQred",
                "K_QAPQ",
                "k_H",
                "k_F",
                "k_P",
                "PSII_total",
            ],
            outputs=["B0", "B1", "B2", "B3"],
        ),
    )
    m.add_readout(
        "Fluo",
        fn=_fluorescence,
        args=["Q", "B0", "B2", "k_H", "k_F", "k_P"],
    )

    m.add_reaction(
        "v_PSII",
        fn=_v_psii,
        args=["B1", "k_P"],
        stoichiometry={
            "pq_red": 1.0,
            "protons": Derived(fn=_two_over, args=["b_H"]),
        },
    )
    m.add_reaction(
        "v_PQ_oxidation",
        fn=_v_pq_oxidation,
        args=["pq_red", "PPFD", "k_Cytb6f", "PQ_total", "K_cytb6f"],
        stoichiometry={
            "pq_red": -1.0,
            "protons": Derived(fn=_four_over, args=["b_H"]),
        },
    )
    m.add_reaction(
        "v_ATPase_activity",
        fn=_v_atpase_activity,
        args=["atpase_activity", "PPFD", "k_ActATPase", "k_DeactATPase"],
        stoichiometry={"atpase_activity": 1.0},
    )
    m.add_reaction(
        "v_ATP_synthase",
        fn=_v_atp_synthase,
        args=[
            "atp",
            "atpase_activity",
            "k_ATPsynthase",
            "K_ATPsynthase",
            "AP_total",
        ],
        stoichiometry={
            "atp": 1.0,
            "protons": Derived(
                fn=_negative_hpr_over,
                args=["HPR", "b_H"],
            ),
        },
    )
    m.add_reaction(
        "v_ATP_consumption",
        fn=_v_atp_consumption,
        args=["atp", "k_ATPconsumption"],
        stoichiometry={"atp": -1.0},
    )
    m.add_reaction(
        "v_KEA3",
        fn=_v_proton_leak,
        args=["protons", "k_leak", "H_stroma"],
        stoichiometry={
            "protons": Derived(fn=_negative_one_over, args=["b_H"]),
        },
    )
    m.add_reaction(
        "v_Dd_deepoxidation",
        fn=_v_dd_deepoxidation,
        args=["dd", "k_deepoxidase"],
        stoichiometry={"dd": -1.0},
    )
    m.add_reaction(
        "v_Dt_epoxidation",
        fn=_v_dt_epoxidation,
        args=["dt", "PPFD", "k_epoxidation", "k_ETC"],
        stoichiometry={"dd": 1.0},
    )

    return m

## 5. PAM light protocol

The protocol alternates measuring or actinic light with 5000 µmol photons m⁻² s⁻¹ saturating pulses, following the timing sequence used in the authors' repository.

In [ ]:
# ------------------------------------------------------------------
# PAM simulation and NPQ calculation
# ------------------------------------------------------------------
def make_pam_protocol(
    actinic_light: float = ACTINIC_LIGHT,
    recovery_light: float = RECOVERY_LIGHT,
) -> pd.DataFrame:
    durations = np.array(
        [1.0, 0.8]
        + [2.0, 0.8]
        + [5.0, 0.8] * 2
        + [15.86666667, 0.8]
        + [30.86666667, 0.8] * 11
        + [15.53333333, 0.8]
        + [30.86666667, 0.8] * 15,
        dtype=float,
    )
    ppfd = np.array(
        [0.0, SATURATING_PULSE] * 2
        + [actinic_light, SATURATING_PULSE] * 14
        + [recovery_light, SATURATING_PULSE] * 16,
        dtype=float,
    )
    end_time = np.cumsum(durations)
    return pd.DataFrame(
        {
            "duration_s": durations,
            "PPFD": ppfd,
            "start_s": end_time - durations,
            "end_s": end_time,
        }
    )

## 6. Dark adaptation

In [ ]:
def _dark_adapt(model: Model, duration_s: float = 1000.0) -> dict[str, float]:
    simulation = (
        Simulator(model)
        .update_parameter("PPFD", 0.0)
        .simulate(duration_s, steps=3000)
        .get_result()
    )
    return simulation.unwrap_or_err().get_new_y0()

## 7. Extract fluorescence maxima and calculate NPQ

In [ ]:
def _extract_pulse_data(
    variables: pd.DataFrame,
    protocol: pd.DataFrame,
) -> pd.DataFrame:
    pulse_rows = protocol.loc[np.isclose(protocol["PPFD"], SATURATING_PULSE)]
    phases = ["dark"] * 2 + ["actinic"] * 14 + ["recovery"] * 16
    records: list[dict[str, float | str]] = []

    for phase, (_, pulse) in zip(phases, pulse_rows.iterrows(), strict=True):
        mask = (variables.index > pulse["start_s"]) & (
            variables.index <= pulse["end_s"] + 1e-9
        )
        pulse_data = variables.loc[mask]
        if pulse_data.empty:
            raise RuntimeError("No simulated values were recorded during a PAM pulse.")

        peak_index = pulse_data["Fluo"].idxmax()
        records.append(
            {
                "phase": phase,
                "time_s": float(peak_index),
                "Fm": float(variables.at[peak_index, "Fluo"]),
                "DES_percent": float(variables.at[peak_index, "dt"] * 100.0),
            }
        )

    peaks = pd.DataFrame(records)
    fm_dark = float(peaks.loc[0, "Fm"])
    peaks["NPQ"] = (fm_dark - peaks["Fm"]) / peaks["Fm"]
    peaks["time_min"] = peaks["time_s"] / 60.0
    return peaks

## 8. Run one complete PAM simulation

In [ ]:
def simulate_pam(
    genotype: str,
    actinic_light: float = ACTINIC_LIGHT,
    recovery_light: float = RECOVERY_LIGHT,
    samples_per_second: float = 5.0,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    model = get_seydoux2022_npq(genotype)
    initial_conditions = _dark_adapt(model)
    protocol = make_pam_protocol(actinic_light, recovery_light)
    simulator = Simulator(model, y0=initial_conditions)

    for _, segment in protocol.iterrows():
        simulator.update_parameter("PPFD", float(segment["PPFD"]))
        steps = max(3, int(np.ceil(segment["duration_s"] * samples_per_second)) + 1)
        simulator.simulate(float(segment["end_s"]), steps=steps)

    simulation = simulator.get_result().unwrap_or_err()
    variables = simulation.variables.copy()
    variables.index.name = "time_s"
    pulse_data = _extract_pulse_data(variables, protocol)
    return variables, pulse_data

## 9. Simulate all three KEA3 conditions

In [ ]:
def simulate_figure_3() -> dict[str, tuple[pd.DataFrame, pd.DataFrame]]:
    return {name: simulate_pam(name) for name in GENOTYPES}

## 10. Light-response simulation for Figure 3c

In [ ]:
def simulate_light_response(
    light_intensities: np.ndarray,
    adaptation_time: float = 600.0,
    samples_per_second: float = 5.0,
) -> pd.DataFrame:
    """Calculate steady-state NPQ at each actinic-light intensity."""
    rows = []

    dark_duration = 1.0
    pulse_duration = 0.8
    relaxation_duration = 2.0

    for genotype in GENOTYPES:
        model = get_seydoux2022_npq(genotype)
        dark_adapted_y0 = _dark_adapt(model)

        for light in np.asarray(light_intensities, dtype=float):
            simulator = Simulator(model, y0=dark_adapted_y0)

            # Dark interval before measuring Fm
            t_dark_end = dark_duration
            simulator.update_parameter("PPFD", 0.0)
            simulator.simulate(
                t_dark_end,
                steps=max(
                    3,
                    int(np.ceil(dark_duration * samples_per_second)) + 1,
                ),
            )

            # Saturating pulse in darkness: Fm
            t_dark_pulse_end = t_dark_end + pulse_duration
            simulator.update_parameter("PPFD", SATURATING_PULSE)
            simulator.simulate(
                t_dark_pulse_end,
                steps=max(
                    3,
                    int(np.ceil(pulse_duration * samples_per_second)) + 1,
                ),
            )

            # Short relaxation after the dark pulse
            t_relaxation_end = t_dark_pulse_end + relaxation_duration
            simulator.update_parameter("PPFD", 0.0)
            simulator.simulate(
                t_relaxation_end,
                steps=max(
                    3,
                    int(np.ceil(relaxation_duration * samples_per_second)) + 1,
                ),
            )

            # Constant actinic light until steady state
            t_light_end = t_relaxation_end + adaptation_time
            simulator.update_parameter("PPFD", float(light))
            simulator.simulate(
                t_light_end,
                steps=max(
                    3,
                    int(np.ceil(adaptation_time * samples_per_second)) + 1,
                ),
            )

            # Saturating pulse after light adaptation: Fm'
            t_light_pulse_end = t_light_end + pulse_duration
            simulator.update_parameter("PPFD", SATURATING_PULSE)
            simulator.simulate(
                t_light_pulse_end,
                steps=max(
                    3,
                    int(np.ceil(pulse_duration * samples_per_second)) + 1,
                ),
            )

            simulation = simulator.get_result().unwrap_or_err()

            variables = simulation.variables.copy()
            variables.index = variables.index.astype(float)
            variables.index.name = "time_s"

            dark_pulse = variables.loc[
                (variables.index > t_dark_end)
                & (variables.index <= t_dark_pulse_end + 1e-9)
            ]

            light_pulse = variables.loc[
                (variables.index > t_light_end)
                & (variables.index <= t_light_pulse_end + 1e-9)
            ]

            if dark_pulse.empty or light_pulse.empty:
                raise RuntimeError(
                    f"Missing pulse output for {genotype} at PPFD={light}."
                )

            fm_dark = float(dark_pulse["Fluo"].max())
            fm_light = float(light_pulse["Fluo"].max())

            npq = (fm_dark - fm_light) / fm_light

            rows.append(
                {
                    "genotype": genotype,
                    "light": float(light),
                    "NPQ": float(max(npq, 0.0)),
                }
            )

    return pd.DataFrame(rows)

## 11. Run the simulations used for Figure 3b and Figure 3c

In [ ]:
# Run the simulations used in Figure 3b and Figure 3c
kinetics = {
    genotype: simulate_pam(
        genotype=genotype,
        actinic_light=200.0,
        recovery_light=0.0,
    )[1]
    for genotype in GENOTYPES
}

light_intensities = np.linspace(0.0, 550.0, 10)

light_response = simulate_light_response(
    light_intensities=light_intensities,
    adaptation_time=600.0,
)

labels = {
    "WT": "WT, K = 2",
    "Mutant 1": r"$kea3$-1,$kea3$-2, K = 10",
    "Overexpressor 1": r"$kea3$-1/KEA3-1eGFP, K = 1.8",
}

## Figure 3b – NPQ kinetics

In [ ]:
# ------------------------------------------------------------------
# F3B – NPQ kinetics
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.2, 4.8))

for genotype in ["WT", "Mutant 1", "Overexpressor 1"]:
    peaks = kinetics[genotype]
    ax.plot(
        peaks["time_min"],
        peaks["NPQ"],
        linewidth=2,
        color=GENOTYPES[genotype].color,
        label=labels[genotype],
    )

ax.set_xlim(0, 17)
ax.set_ylim(0, 2)
ax.set_xlabel("Time (min)")
ax.set_ylabel("NPQ")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

## Figure 3c – NPQ versus light intensity

In [ ]:
# ------------------------------------------------------------------
# F3C – NPQ versus light intensity
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.2, 4.8))

for genotype in ["WT", "Mutant 1", "Overexpressor 1"]:
    curve = light_response.loc[light_response["genotype"] == genotype]
    ax.plot(
        curve["light"],
        curve["NPQ"],
        linewidth=2,
        color=GENOTYPES[genotype].color,
        label=labels[genotype],
    )

ax.set_xlim(0, 550)
ax.set_ylim(0, 3)
ax.set_xlabel(r"Light intensity ($\mu$mol photons m$^{-2}$ s$^{-1}$)")
ax.set_ylabel("NPQ")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()